# HeyTaxi
Extracting insights about taxi rides.

In [0]:
spark.sql("USE CATALOG heytaxi")


DB_NAME = "nyc_taxi_pipeline"
spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}")
spark.sql(f"USE {DB_NAME}")

display(f"Active database: {DB_NAME}")

# Bronze Level

In [0]:
import urllib.request

# 1. Create a Volume if you don't have one already (or use your existing catalog/schema/volume)
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.nyc_tlc_volume")

# 2. Path to store the parquet file in the Volume
volume_file_path = "/Volumes/workspace/default/nyc_tlc_volume/yellow_tripdata_2024-01.parquet"
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"

print("Downloading parquet file into Volume...")
urllib.request.urlretrieve(url, volume_file_path)

# 3. Read directly into PySpark (No 'file:' prefix required for Volumes!)
df_raw = spark.read.parquet(volume_file_path)

print(f"Success! Total records read: {df_raw.count():,}")

In [0]:
display(df_raw)

In [0]:
from pyspark.sql.functions import current_timestamp, col

# Add audit metadata using Unity Catalog-compatible _metadata.file_path
df_bronze = df_raw \
    .withColumn("ingestion_time", current_timestamp()) \
    .withColumn("source_file", col("_metadata.file_path"))

# Save as Bronze Delta Table
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_yellow_taxi")



In [0]:
print("Bronze Layer complete! Total records:", spark.read.table("bronze_yellow_taxi").count())

In [0]:
%sql
SELECT * FROM bronze_yellow_taxi LIMIT 10;

#Silver Level

In [0]:
from pyspark.sql.functions import col, unix_timestamp, round as spark_round

# 1. Read raw data from the Bronze table
df_bronze = spark.read.table("bronze_yellow_taxi")

display(df_bronze)

In [0]:


# 2. Apply cleaning rules and feature engineering
df_silver = df_bronze \
    .filter(
        # Date boundary check (removes rogue dates outside Jan 2024)
        (col("tpep_pickup_datetime") >= "2024-01-01") & 
        (col("tpep_pickup_datetime") < "2024-02-01") &
        # Business logic checks
        (col("passenger_count") > 0) &
        (col("trip_distance") > 0.0) &
        (col("fare_amount") > 0.0)
    ) \
    .withColumn(
        "trip_duration_minutes", 
        spark_round((unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60, 2)
    ) \
    .filter(
        # Keep realistic trip durations (between 1 min and 24 hours)
        (col("trip_duration_minutes") >= 1.0) & 
        (col("trip_duration_minutes") <= 1440.0)
    )

# 3. Write cleaned data to the Silver Delta Table
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_yellow_taxi")

print(f"Silver Layer processing complete!")



In [0]:
bronze_count = spark.read.table("bronze_yellow_taxi").count()
silver_count = spark.read.table("silver_yellow_taxi").count()

print(f"Bronze Raw Records  : {bronze_count:,}")
print(f"Silver Clean Records: {silver_count:,}")
print(f"Filtered Out Rows   : {bronze_count - silver_count:,}")

display(df_silver)

In [0]:
display(
    spark.read.table("silver_yellow_taxi")
    .select(
        "tpep_pickup_datetime", 
        "tpep_dropoff_datetime", 
        "trip_duration_minutes", 
        "trip_distance", 
        "fare_amount", 
        "total_amount"
    )
    .limit(10)
)

# Gold Level

In [0]:
# 1. Read cleaned data from Silver table
df_silver = spark.read.table("silver_yellow_taxi")

display(df_silver)

In [0]:
from pyspark.sql.functions import count, avg, sum as spark_sum, round as spark_round

# 1. Read cleaned data from Silver table
df_silver = spark.read.table("silver_yellow_taxi")

# 2. Build Business Aggregations
df_gold_location = df_silver.groupBy("PULocationID") \
    .agg(
        count("*").alias("total_trips"),
        spark_round(avg("trip_distance"), 2).alias("avg_distance_miles"),
        spark_round(avg("fare_amount"), 2).alias("avg_fare_amount"),
        spark_round(avg("trip_duration_minutes"), 2).alias("avg_duration_minutes"),
        spark_round(spark_sum("total_amount"), 2).alias("total_revenue")
    )

# 3. Save as Gold Delta Table
df_gold_location.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_location_performance")

print("Gold Layer Data Mart complete!")

## Query gold database

In [0]:

%sql
SELECT *
FROM gold_location_performance
ORDER BY total_trips DESC
LIMIT 10;

In [0]:
%sql
SHOW TABLES;